# Day 8 — XGBoost Baseline on SMOTE-Resampled Data

## Objective

Evaluate a default XGBoost classifier trained on SMOTE-resampled
training data while evaluating exclusively on the untouched
time-based test set.

## Experimental Pipeline

Raw data
    ↓
Time-based train/test split
    ↓
SMOTE on training data only
    ↓
XGBoost with default parameters
    ↓
Untouched test set
    ↓
Precision / Recall / F1 / Average Precision (AUC-PR)

## Primary Metric

Average Precision (AUC-PR)

## Secondary Metrics

Precision
Recall
F1

In [1]:
from pathlib import Path


import pandas as pd
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

Imports

In [2]:
RAW_DATA_PATH = Path("data/creditcard_1.csv")

TARGET_COLUMN = "Class"

RANDOM_STATE = 42

In [3]:

df = pd.read_csv(RAW_DATA_PATH)
df = df.sort_values("Time").reset_index(drop = True)

target_column = "Class"

split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

X_train = train_df.drop(columns=[target_column])
y_train = train_df[target_column]

X_test = test_df.drop(columns=[target_column])
y_test = test_df[target_column]

### Smote

In [4]:
smote = SMOTE(
    sampling_strategy="auto",
    random_state=42
)

X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train,
    y_train
)

In [5]:
test_shape_before = X_test.shape

test_class_counts_before = (
    y_test.value_counts()
    .sort_index()
    .to_dict()
)

test_class_percentages_before = (
    y_test.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .to_dict()
)
test_shape_after = X_test.shape

test_class_counts_after = (
    y_test.value_counts()
    .sort_index()
    .to_dict()
)

test_class_percentages_after = (
    y_test.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .to_dict()
)

assert test_shape_before == test_shape_after
assert test_class_counts_before == test_class_counts_after
assert test_class_percentages_before == test_class_percentages_after

print("✓ Test-set integrity verified.")

✓ Test-set integrity verified.


### Model Training

In [6]:
xgb_model = XGBClassifier(
    random_state=RANDOM_STATE
)
## Training
xgb_model.fit(
    X_train_resampled,
    y_train_resampled
)
## Predictions
y_pred = xgb_model.predict(X_test)

In [7]:
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

In [8]:
y_pred_proba

array([5.30230238e-07, 1.56545340e-07, 1.00335896e-07, ...,
       3.22326514e-07, 3.12470820e-06, 3.91479489e-06], dtype=float32)

In [9]:
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
average_precision = average_precision_score(
    y_test,
    y_pred_proba
)

In [10]:
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1: {f1:.4f}")
print(f"AUC-PR / Average Precision: {average_precision:.4f}")

Precision: 0.7532
Recall: 0.7733
F1: 0.7632
AUC-PR / Average Precision: 0.7999


In [11]:
print("Test observations:", len(X_test))
print("Predictions:", len(y_pred))
print("Probabilities:", len(y_pred_proba))

Test observations: 56962
Predictions: 56962
Probabilities: 56962
